# Experimento B — Paralelismo a nivel de tareas

Este notebook compara tres estrategias de ejecución de **lotes independientes** de feature engineering, evaluando el efecto del **GIL** (Global Interpreter Lock) de Python según el tipo de carga:

1. **Secuencial** — ejecutar los lotes uno tras otro en un solo hilo.
2. **ThreadPoolExecutor** — paralelismo basado en hilos, comparten memoria pero quedan limitados por el GIL.
3. **ProcessPoolExecutor** — paralelismo basado en procesos, cada uno con su propio intérprete y GIL independiente.

**Dos cargas distintas** se evalúan en paralelo para mostrar cómo la estrategia óptima cambia según la naturaleza del trabajo:

- **CPU-bound:** limpieza de pares (nombre, email) con normalización, validación, detección de coherencia y hashing MD5. Toda la operación es cálculo puro, sin esperas externas.
- **I/O-bound:** ciclo de write/read/delete sobre archivos temporales reales en disco. La mayor parte del tiempo el programa espera al sistema operativo.

**Métricas medidas:** tiempo total, speedup respecto a la secuencial y uso de CPU.

## 1. Importación de librerías

Imports necesarios para el experimento:

- `numpy`, `pandas`, `matplotlib`, `seaborn` — análisis y visualización (igual que en el experimento A).
- `time`, `psutil` — medición de tiempos y uso de CPU.
- `pathlib.Path` — manejo de rutas portable.
- `random`, `string` — generación de datos aleatorios.
- `faker` — librería estándar de generación de datos sintéticos realistas (nombres, emails, etc.).
- `unicodedata` — normalización de caracteres acentuados.
- `concurrent.futures.ThreadPoolExecutor` y `ProcessPoolExecutor` — APIs modernas de paralelismo en Python.
- `worker_B.procesar_lote_cpu` y `worker_B.procesar_lote_io` — funciones de procesamiento definidas en archivo separado (requerido por `multiprocessing` en macOS, que usa el método `spawn`).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import psutil
from pathlib import Path
import random
import string
from faker import Faker
import unicodedata
from worker_B import procesar_lote_cpu
from worker_B import procesar_lote_io
from concurrent.futures import ThreadPoolExecutor
from concurrent.futures import ProcessPoolExecutor

## 2. Configuración de rutas

Se definen las rutas para guardar los datos generados (CSVs con mediciones) y las visualizaciones (PNGs). El notebook vive en `experimentos/`, por lo que se sube un nivel para acceder a `datos/` y `visuali/`.

In [ ]:
RAIZ = Path('..').resolve()
RUTA_DATOS = RAIZ / 'datos'
RUTA_VISUALI = RAIZ / 'visuali'

## 3. Inicialización de Faker

Se inicializa el generador `Faker` con localización chilena (`es_CL`) y semilla fija para reproducibilidad. La lista `DOMINIOS` define los dominios de email que se usarán al construir emails coherentes con los nombres.

Las llamadas a `fake.name()`, `fake.email()`, etc. en esta celda son simplemente para verificar que Faker está funcionando. La generación real ocurre en `generar_datos_cpu`.

In [ ]:
Faker.seed(3)
fake = Faker('es_CL')
fake.name()
fake.email()
fake.first_name()
fake.last_name()
DOMINIOS = ['gmail.com', 'hotmail.com', 'yahoo.com', 'outlook.com']
dominio = random.choice(DOMINIOS)

## 4. Funciones auxiliares y generadores de datos

Esta celda agrupa las funciones de preparación del experimento.

### `ensuciar(texto)`
Aplica suciedad aleatoria a un string para simular datos del mundo real con inconsistencias. Probabilísticamente:
- Agrega espacios al inicio (40%) y al final (40%).
- Mezcla mayúsculas y minúsculas en cada letra (30%).

Esto da variedad al dataset para que la función de limpieza tenga trabajo real que hacer.

### `quitar_acentos(texto)`
Elimina acentos y caracteres no-ASCII usando descomposición Unicode. Por ejemplo, `"María Núñez"` → `"Maria Nunez"`. Necesario para construir emails válidos a partir de nombres con acentos.

### `email_desde_nombre(nombre)`
Genera un email **coherente** con un nombre. Por ejemplo: `"María García López"` → `"maria.lopez@gmail.com"`. Los emails coherentes representan el ~70% del dataset.

### `generar_datos_cpu(n, seed=3)`
Genera el dataset de la carga CPU-bound: lista de tuplas `(nombre_sucio, email_sucio)`. Para cada par:
- 70% probabilidad de email **coherente** (derivado del nombre).
- 30% probabilidad de email **incoherente** (`fake.email()` aleatorio).
- Tanto nombre como email se ensucian antes de retornarlos.

Esta variedad es lo que hace que la detección de coherencia (en `procesar_lote_cpu`) sea una feature significativa.

### `generar_datos_io(n, seed=1)`
Genera el dataset I/O-bound: simplemente `list(range(n))`. La carga I/O no necesita datos sintéticos sucios — solo identificadores que la función `procesar_lote_io` usará para crear archivos.

### `medir(ejecutor_funcion, *args, **kwargs)`
Función para medir tiempo de ejecución y uso de CPU. El uso de `*args, **kwargs` permite que la misma función sirva tanto para `ejecutar_secuencial(funcion, lotes)` como para `ejecutar_threads(funcion, lotes, n_workers)`, pasando solo los argumentos que cada ejecutora necesita.

In [ ]:
def ensuciar(texto):
    if random.random() < 0.4:
        texto = "  " + texto
    if random.random() < 0.4:
        texto = texto + "  "
    if random.random() < 0.3:
        nuevo_texto = "  "
        for letra in texto:
            if random.random() < 0.5:
                nuevo_texto = nuevo_texto + letra.upper()
            else:
                nuevo_texto = nuevo_texto + letra.lower()
        texto = nuevo_texto
    return texto

def quitar_acentos(texto):
    # NFKD descompone los caracteres acentuados en (letra + acento)
    nfkd = unicodedata.normalize('NFKD', texto)
    # Filtramos solo los caracteres ASCII (los acentos no son ASCII)
    return ''.join(c for c in nfkd if not unicodedata.combining(c))

def email_desde_nombre(nombre):
    nombre = quitar_acentos(nombre).lower()
    dominio = random.choice(DOMINIOS)
    
    partes = nombre.split()
    if len(partes) >= 2:
        primer_nombre = partes[0]
        apellido = partes[-1]
        return f"{primer_nombre}.{apellido}@{dominio}"
    else:
        return f"{nombre}@{dominio}"
    
def generar_datos_cpu(n, seed=3):
    Faker.seed(seed)
    random.seed(seed)
    datos = []
    for i in range(n):
        nombre = fake.name()
        # 70% coherente, 30% incoherente
        if random.random() < 0.7:
            email = email_desde_nombre(nombre)
        else:
            email = fake.email()
        nombre_sucio = ensuciar(nombre)
        email_sucio = ensuciar(email)
        datos.append((nombre_sucio, email_sucio))
    return datos

def generar_datos_io(n, seed=1):
    return list(range(n))

def medir(ejecutor_funcion, *args, **kwargs):
    psutil.cpu_percent(interval=None)         # inicializa contador
    inicio = time.perf_counter()
    resultado = ejecutor_funcion(*args, **kwargs)
    fin = time.perf_counter()
    cpu_pct = psutil.cpu_percent(interval=None)
    return fin - inicio, cpu_pct

## 5. Funciones ejecutoras y división en lotes

Esta celda define las **tres estrategias de ejecución** que se compararán, más una función auxiliar para dividir el dataset en lotes.

### `ejecutar_secuencial(funcion, lotes)`
Procesa los lotes uno tras otro en el hilo principal. No usa workers. Es el **baseline** contra el cual se calcula el speedup. Su tiempo de ejecución es la "línea base" de cada carga.

### `ejecutar_threads(funcion, lotes, n_workers)`
Usa `ThreadPoolExecutor` para distribuir los lotes entre `n_workers` hilos del mismo proceso.
- **Ventaja:** los hilos comparten memoria, son livianos de crear.
- **Desventaja:** el GIL impide ejecución Python simultánea — solo un hilo avanza a la vez en código Python puro.
- **Caso favorable:** I/O-bound. Cuando un hilo espera al sistema operativo, libera el GIL y otro hilo avanza.
- **Caso desfavorable:** CPU-bound. El GIL serializa la ejecución, anulando la ventaja de tener múltiples hilos.

### `ejecutar_processes(funcion, lotes, n_workers)`
Usa `ProcessPoolExecutor` para distribuir los lotes entre `n_workers` procesos independientes. Cada proceso tiene su propio intérprete Python y su propio GIL, así que pueden ejecutar bytecode Python en paralelo real.
- **Ventaja:** sortea el GIL completamente.
- **Desventaja:** crear procesos es más caro que crear hilos, y la comunicación entre procesos requiere serialización (pickle).
- **Caso favorable:** CPU-bound puro.
- **Caso menos favorable:** I/O-bound. Funciona, pero los hilos suelen ser más eficientes para esto por su menor overhead.

### `dividir_en_lotes(datos, tamano_lote)`
Divide una lista grande en sublistas del tamaño especificado. Usa list comprehension con slicing. Si el último chunk es más pequeño, lo deja igual (no rellena).

In [ ]:
def ejecutar_secuencial(funcion, lotes):
    resultados = []
    for lote in lotes:
        resultados.extend(funcion(lote))
    return resultados

def ejecutar_threads(funcion, lotes, n_workers):
    resultados = []
    with ThreadPoolExecutor(max_workers=n_workers) as executor:
        for resultado_lote in executor.map(funcion, lotes):
            resultados.extend(resultado_lote)
    return resultados

def ejecutar_processes(funcion, lotes, n_workers):
    resultados = []
    with ProcessPoolExecutor(max_workers=n_workers) as executor:
        for resultado_lote in executor.map(funcion, lotes):
            resultados.extend(resultado_lote)
    return resultados

def dividir_en_lotes(datos, tamano_lote):
    return [datos[i:i+tamano_lote] for i in range(0, len(datos), tamano_lote)]

## 6. Parámetros del experimento

Se definen los parámetros que controlan el barrido experimental:

- **`TOTAL_ITEMS_CPU = 300_000`**: tamaño del dataset CPU-bound. Más grande que el I/O porque el procesamiento por item es relativamente rápido y se necesita volumen para que las diferencias entre estrategias sean medibles.
- **`TOTAL_ITEMS_IO = 3_000`**: tamaño del dataset I/O-bound. Más chico porque cada item implica write+read+delete reales en disco, que son operaciones lentas.
- **`TAMANOS_LOTE = [200, 1000, 5000]`**: tres tamaños de chunk. Variar este parámetro permite estudiar cómo la **granularidad** afecta el rendimiento.
- **`N_REPETICIONES = 3`**: 3 repeticiones por configuración para reportar promedio y desviación.
- **`configuraciones`**: lista de las 7 estrategias a probar (1 secuencial + 3 configuraciones de threads + 3 de processes). La secuencial solo aparece una vez con `n_workers=1` porque no se beneficia de variar workers.

**Total de mediciones**: 2 cargas × 3 tamaños × 7 configuraciones × 3 reps = **126 mediciones**.

In [ ]:
TOTAL_ITEMS_CPU = 300000
TOTAL_ITEMS_IO = 3_000
TAMANOS_LOTE = [200, 1000, 5000]
N_REPETICIONES = 3

configuraciones = [
    ('secuencial', 1),
    ('threads', 2),
    ('threads', 4),
    ('threads', 8),
    ('processes', 2),
    ('processes', 4),
    ('processes', 8),
]

## 7. Loop experimental principal

Este es el corazón del experimento. Cuatro loops anidados recorren todas las combinaciones:

```
para cada carga (cpu, io):
    elegir dataset y función según carga
    para cada tamaño_lote:
        dividir el dataset en lotes
        para cada (ejecutor, n_workers) en configuraciones:
            para cada repetición (1 a 3):
                medir y guardar
```

**Detalles importantes**:

- Los datasets se generan **una sola vez** al inicio (fuera del loop). Generarlos en cada iteración sería redundante y costoso.
- El `if/elif/else` interno selecciona la ejecutora correcta según el nombre. La secuencial ignora `n_workers` (siempre es 1).
- Cada medición se guarda como un **diccionario** en la lista `resultados`, con todas las dimensiones del experimento.
- El `print` permite monitorear el progreso durante la corrida (que dura varios minutos).

In [ ]:
# Generar datasets una sola vez
datos_cpu = generar_datos_cpu(TOTAL_ITEMS_CPU, seed=1)
datos_io = generar_datos_io(TOTAL_ITEMS_IO, seed=1)

resultados = []

for carga in ['cpu', 'io']:
    # Elegir dataset y función según carga
    if carga == 'cpu':
        dataset = datos_cpu
        funcion = procesar_lote_cpu
    else:
        dataset = datos_io
        funcion = procesar_lote_io
    
    for tamano_lote in TAMANOS_LOTE:
        # Dividir el dataset en lotes
        lotes = dividir_en_lotes(dataset, tamano_lote)
        
        for ejecutor, n_workers in configuraciones:
            for rep in range(N_REPETICIONES):
                # Llamar a la ejecutora correcta y medir
                if ejecutor == 'secuencial':
                    tiempo, cpu = medir(ejecutar_secuencial, funcion, lotes)
                elif ejecutor == 'threads':
                    tiempo, cpu = medir(ejecutar_threads, funcion, lotes, n_workers)
                else:
                    tiempo, cpu = medir(ejecutar_processes, funcion, lotes, n_workers)
                
                # Guardar el resultado
                resultados.append({
                    'carga': carga,
                    'ejecutor': ejecutor,
                    'n_workers': n_workers,
                    'tamano_lote': tamano_lote,
                    'repeticion': rep + 1,
                    'tiempo': tiempo,
                    'cpu_pct': cpu,
                })
                
                print(f"  {carga} | lote={tamano_lote} | {ejecutor}({n_workers}) rep {rep+1}: {tiempo:.3f}s")

print(f"\nTotal de mediciones: {len(resultados)}")

## 8. Construcción del DataFrame y guardado

Las 126 mediciones individuales se convierten en un DataFrame de Pandas y se guardan a CSV en la carpeta `datos/`. El archivo `experimentoB_mediciones.csv` contiene **todas las mediciones crudas**, útil si se necesita mostrar dispersión entre repeticiones o re-analizar los datos.

In [ ]:
df_b = pd.DataFrame(resultados)

# Guardar CSV
ruta_csv = RUTA_DATOS / 'experimentoB_mediciones.csv'
df_b.to_csv(ruta_csv, index=False)
print(f"Guardado: {ruta_csv}")
print(f"Total de filas: {len(df_b)}")
df_b.head()

## 9. Estadísticas agregadas

Se agrupan las mediciones por (carga, ejecutor, n_workers, tamano_lote) y se calculan:

- **`tiempo_promedio`**: media de los tiempos en las 3 repeticiones.
- **`tiempo_std`**: desviación estándar entre las 3 repeticiones (variabilidad de la medición).
- **`cpu_promedio`**: promedio del uso de CPU.

Esta tabla agregada se usa como input para todas las visualizaciones posteriores. Se guarda también a CSV (`experimentoB_estadisticas.csv`) para que pueda usarse externamente.

In [ ]:
df_stats_b = df_b.groupby(['carga', 'ejecutor', 'n_workers', 'tamano_lote']).agg(
    tiempo_promedio=('tiempo', 'mean'),
    tiempo_std=('tiempo', 'std'),
    cpu_promedio=('cpu_pct', 'mean'),
).reset_index()

df_stats_b.to_csv(RUTA_DATOS / 'experimentoB_estadisticas.csv', index=False)
print(df_stats_b)

## 10. Visualización 1: Tiempo de ejecución vs número de workers

Gráfico de **dos paneles** lado a lado (uno por carga). Cada panel muestra el tiempo promedio de ejecución de las tres estrategias (secuencial, threads, processes) en función del número de workers.

**Detalle de preparación de datos**: se promedia entre los tres tamaños de lote para simplificar el gráfico (el efecto del tamaño de lote se analiza en el gráfico 3). Esto se hace con un `groupby` adicional sobre `df_stats_b`.

**Lo que se observa:**

- **Panel CPU-bound**: la línea de threads queda **pegada a la secuencial** (no aporta), confirmando el efecto del GIL. La línea de processes desciende claramente con más workers, mostrando paralelismo real.
- **Panel I/O-bound**: ahora **ambas estrategias paralelas bajan**, e incluso threads supera levemente a processes. El GIL se libera durante operaciones I/O, permitiendo que múltiples threads avancen simultáneamente.

Esta inversión de comportamiento entre los dos paneles es **el aprendizaje central del experimento**.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Promediar entre tamaños de lote
df_g1 = df_stats_b.groupby(['carga', 'ejecutor', 'n_workers']).agg(
    tiempo=('tiempo_promedio', 'mean')
).reset_index()

cargas = ['cpu', 'io']
titulos = ['CPU-bound (limpieza de strings)', 'I/O-bound (read/write archivos)']

for i, carga in enumerate(cargas):
    subset_carga = df_g1[df_g1['carga'] == carga]
    
    for ejecutor in ['secuencial', 'threads', 'processes']:
        subset = subset_carga[subset_carga['ejecutor'] == ejecutor]
        axes[i].plot(subset['n_workers'], subset['tiempo'], 
                     marker='o', label=ejecutor)
    
    axes[i].set_title(titulos[i])
    axes[i].set_xlabel('Número de workers')
    axes[i].set_ylabel('Tiempo (s)')
    axes[i].legend()
    axes[i].grid(True, alpha=0.3)
    axes[i].set_xticks([1, 2, 4, 8])

fig.suptitle('Tiempo de ejecución según ejecutor y workers – Experimento B', 
             fontsize=13)
plt.tight_layout()
plt.savefig(RUTA_VISUALI / 'experimentoB_tiempo_vs_workers.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Visualización 2: Speedup vs número de workers

Mismo layout de dos paneles, pero ahora el eje Y muestra el **speedup** respecto a la versión secuencial. El speedup se calcula como `tiempo_secuencial / tiempo_estrategia`.

**Cálculo del speedup**: se usa `apply` con función lambda para dividir cada fila por el tiempo secuencial **de su misma carga**. Esto se logra mapeando `'cpu'` y `'io'` a sus tiempos secuenciales correspondientes mediante `set_index('carga')`.

**Líneas de referencia importantes**:

- **`y = 1` (gris discontinua)**: speedup neutro. Por debajo de esta línea, paralelizar es contraproducente.
- **Diagonal punteada negra**: speedup ideal lineal (con N workers, speedup = N). Esto representa la promesa "teórica" del paralelismo perfecto.

**Lo que se observa:**

- **Panel CPU-bound**: threads se mantiene en y=1 (ningún speedup), confirmando el GIL. Processes alcanza ~3.46× con 8 workers, pero está **muy por debajo del ideal lineal de 8×**.
- **Panel I/O-bound**: ambas estrategias dan speedup modesto (~1.4×), también muy lejos del ideal. Esto sugiere que el cuello de botella es la **contención del subsistema de almacenamiento**: por más workers que pongamos, el SSD solo puede atender un número limitado de operaciones simultáneas.

Esta gran distancia entre speedup observado e ideal es precisamente el "**error de asumir speedup lineal**" que la pauta exige discutir explícitamente.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Calcular speedup respecto a la secuencial de cada carga
tiempo_seq_por_carga = df_g1[df_g1['ejecutor'] == 'secuencial'].set_index('carga')['tiempo']
df_g1['speedup'] = df_g1.apply(
    lambda fila: tiempo_seq_por_carga[fila['carga']] / fila['tiempo'],
    axis=1
)

cargas = ['cpu', 'io']
titulos = ['CPU-bound', 'I/O-bound']

for i, carga in enumerate(cargas):
    subset_carga = df_g1[df_g1['carga'] == carga]    
    for ejecutor in ['threads', 'processes']:
        subset = subset_carga[subset_carga['ejecutor'] == ejecutor]
        axes[i].plot(subset['n_workers'], subset['speedup'],
                     marker='o', label=ejecutor)    
    
    # Líneas de referencia
    axes[i].axhline(y=1, color='gray', linestyle='--', label='Sin speedup')
    axes[i].plot([1, 2, 4, 8], [1, 2, 4, 8], 'k:', label='Speedup ideal lineal')
    
    axes[i].set_title(titulos[i])
    axes[i].set_xlabel('Número de workers')
    axes[i].set_ylabel('Speedup (×)')
    axes[i].set_xticks([1, 2, 4, 8])
    axes[i].legend()
    axes[i].grid(True, alpha=0.3)

fig.suptitle('Speedup según ejecutor y workers – Experimento B', fontsize=13)
plt.tight_layout()
plt.savefig(RUTA_VISUALI / 'experimentoB_speedup.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Visualización 3: Efecto del tamaño de lote

Este gráfico fija el número de workers en **4** (donde se observó el peak en gráficos anteriores) y varía el tamaño de lote. Permite estudiar cómo la **granularidad** del trabajo afecta el rendimiento.

**Detalle de filtrado**: como la versión secuencial siempre tiene `n_workers=1`, se excluye del filtro `n_workers == 4`. Para threads y processes sí se aplica el filtro.

**Lo que se observa:**

- **CPU-bound**: la versión con processes muestra mejor rendimiento con lotes pequeños (200 ítems). Con lotes más grandes el tiempo aumenta porque hay menos lotes para distribuir entre los workers, generando desbalance.

- **I/O-bound**: el efecto es más dramático. Con lote=5000 sobre un dataset de solo 3000 ítems, **solo se genera 1 lote**, por lo que 3 de los 4 workers quedan ociosos. El paralelismo desaparece y los tiempos se acercan al secuencial.

Esto evidencia un fenómeno importante: **el tamaño de lote óptimo no es trivial**. Lotes muy grandes reducen la cantidad de unidades de trabajo y limitan la paralelización; lotes muy pequeños pueden generar overhead de coordinación. La elección depende del tamaño total del dataset y del número de workers disponibles.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cargas = ['cpu', 'io']
titulos = ['CPU-bound', 'I/O-bound']

for i, carga in enumerate(cargas):
    subset_carga = df_stats_b[df_stats_b['carga'] == carga]
    for ejecutor in ['secuencial', 'threads', 'processes']:
        subset = subset_carga[subset_carga['ejecutor'] == ejecutor]
        # Para threads y processes, filtrar por n_workers=4
        if ejecutor != 'secuencial':
            subset = subset[subset['n_workers'] == 4]
        axes[i].plot(subset['tamano_lote'], subset['tiempo_promedio'],
                     marker='o', label=ejecutor) 
    axes[i].set_title(titulos[i])
    axes[i].set_xlabel('Tamaño de lote')
    axes[i].set_ylabel('Tiempo (s)')
    axes[i].set_xticks([200, 1000, 5000])
    axes[i].legend()
    axes[i].grid(True, alpha=0.3)

fig.suptitle('Efecto del tamaño de lote (4 workers) – Experimento B', fontsize=13)
plt.tight_layout()
plt.savefig(RUTA_VISUALI / 'experimentoB_tamano_lote.png', dpi=150, bbox_inches='tight')
plt.show()

## 13. Visualización 4: Mejor tiempo logrado por estrategia

Gráfico de barras que resume el mejor tiempo alcanzado por cada estrategia, en cada carga. Para cada combinación (carga, ejecutor) se selecciona el mínimo `tiempo_promedio` entre todas las configuraciones de workers y tamaño de lote.

**Detalle**: se usa `groupby('carga', 'ejecutor')['tiempo_promedio'].min()` para obtener el mejor caso de cada estrategia. Luego se reordena explícitamente para que las barras aparezcan siempre en el orden secuencial → threads → processes (en lugar del orden alfabético por defecto). El valor numérico se anota encima de cada barra para lectura directa.

**Lo que se observa:**

- **CPU-bound**: processes mejora drásticamente sobre secuencial (~3.5× más rápido). Threads NO mejora — su mejor tiempo es prácticamente igual al secuencial, confirmando una vez más el efecto del GIL.
- **I/O-bound**: threads es el ganador (1.77× sobre secuencial), processes queda ligeramente atrás. Ambos paralelos mejoran el secuencial.

**Recomendación práctica**: la estrategia óptima depende del tipo de carga.
- Tarea CPU-bound → `ProcessPoolExecutor`.
- Tarea I/O-bound → `ThreadPoolExecutor`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Calcular el mejor tiempo por (carga, ejecutor)
df_mejor = df_stats_b.groupby(['carga', 'ejecutor'])['tiempo_promedio'].min().reset_index()

cargas = ['cpu', 'io']
titulos = ['CPU-bound', 'I/O-bound']
colores = ['#1f77b4', '#ff7f0e', '#2ca02c']  # azul, naranja, verde

for i, carga in enumerate(cargas):
    subset = df_mejor[df_mejor['carga'] == carga]
    
    # Forzar orden: secuencial, threads, processes
    orden = ['secuencial', 'threads', 'processes']
    subset = subset.set_index('ejecutor').loc[orden].reset_index()
    
    barras = axes[i].bar(subset['ejecutor'], subset['tiempo_promedio'], color=colores)
    
    # Anotar el valor encima de cada barra
    for barra, tiempo in zip(barras, subset['tiempo_promedio']):
        axes[i].text(
            barra.get_x() + barra.get_width() / 2,
            barra.get_height(),
            f'{tiempo:.3f}s',
            ha='center', va='bottom'
        )
    
    axes[i].set_title(titulos[i])
    axes[i].set_ylabel('Tiempo (s)')
    axes[i].grid(True, alpha=0.3, axis='y')

fig.suptitle('Mejor tiempo logrado por estrategia – Experimento B', fontsize=13)
plt.tight_layout()
plt.savefig(RUTA_VISUALI / 'experimentoB_mejor_estrategia.png', dpi=150, bbox_inches='tight')
plt.show()

---

# Análisis de resultados 

## Resumen de hallazgos principales

El experimento B demuestra empíricamente que **la estrategia óptima de paralelización depende del tipo de carga**, y que el GIL de Python es el factor central que explica esta diferencia. Los resultados también evidencian que el speedup nunca alcanza el ideal lineal, debido a varios factores que se discuten a continuación.

## Fenómeno 1: El GIL serializa los threads en CPU-bound

**Observación**: en la carga CPU-bound, el speedup de `ThreadPoolExecutor` es ~1.0× independientemente del número de workers (2, 4, 8). El tiempo de ejecución con threads es prácticamente igual al de la versión secuencial.

**Explicación técnica**: el **GIL (Global Interpreter Lock)** de CPython es un candado global que impide que dos hilos ejecuten bytecode Python simultáneamente dentro del mismo proceso. Aunque el `ThreadPoolExecutor` crea múltiples hilos, solo uno puede tener el GIL en cualquier momento. Para tareas que son cómputo Python puro (limpieza de strings, hashing MD5, validación con regex), agregar más hilos no acelera nada — solo agrega overhead de cambio de contexto.

**Implicación práctica**: usar threads para tareas CPU-bound en Python es un anti-patrón. Es uno de los errores más comunes al introducirse al paralelismo en este lenguaje.

## Fenómeno 2: Processes sortea el GIL y obtiene speedup real

**Observación**: en la misma carga CPU-bound, `ProcessPoolExecutor` alcanza speedups de hasta ~3.46× con 8 workers. La curva de tiempo desciende claramente con más procesos.

**Explicación técnica**: cada proceso creado por `multiprocessing` tiene su propio intérprete Python, su propio GIL y su propia memoria. Los procesos pueden ejecutar bytecode Python verdaderamente en paralelo, aprovechando múltiples cores físicos del CPU.

**Costos asociados**: este beneficio no es gratuito. Crear procesos es significativamente más caro que crear threads (en macOS con `spawn`, ~150-200ms por proceso). Además, los datos enviados a los workers deben serializarse con `pickle`, lo que agrega overhead. Estos costos se amortizan cuando la cantidad de trabajo por worker es lo suficientemente grande.

## Fenómeno 3: Threads ganan en I/O-bound

**Observación**: en la carga I/O-bound, threads alcanza el mejor tiempo (0.166s), ligeramente mejor que processes (0.192s). Ambos superan claramente al secuencial (0.294s).

**Explicación técnica**: durante operaciones de I/O (leer/escribir disco, llamadas de red), Python **libera el GIL** mientras espera al sistema operativo. Esto permite que otros threads avancen mientras uno está bloqueado esperando. Como el cuello de botella ya no es el cómputo Python sino la espera externa, el GIL deja de ser una limitación.

**Por qué threads supera a processes**: los procesos también funcionan para I/O, pero pagan el costo extra de creación y serialización sin ganar nada adicional (el GIL ya no era el problema). Los threads son más livianos para este caso.

## Fenómeno 4: Speedup nunca alcanza el ideal lineal

**Observación**: el speedup máximo observado fue ~3.46× con 8 workers en CPU-bound. El ideal lineal sería 8×, lo que indica una eficiencia de solo ~43%. En I/O-bound el speedup máximo fue ~1.77× con 4 workers (eficiencia ~44%).

**Explicación**: el "error de asumir speedup lineal" se debe a múltiples factores que el experimento B evidencia:

1. **Overhead de creación de workers**: crear un proceso toma ~150-200ms. Crear un thread también tiene un costo, aunque mucho menor.
2. **Serialización (pickle)**: enviar datos a los workers de procesos requiere convertirlos a bytes y de vuelta. Para datasets grandes este costo no es despreciable.
3. **Ley de Amdahl**: siempre hay una fracción inherentemente serial (la propia coordinación de los workers, la recolección de resultados). Esa parte no se beneficia del paralelismo.
4. **Contención de recursos**: en I/O-bound, los workers compiten por el mismo dispositivo SSD. Por más workers que pongamos, el subsistema de almacenamiento solo puede atender un número limitado de operaciones simultáneas.
5. **Cores heterogéneos del Apple M4**: el procesador mezcla cores P (rendimiento) y cores E (eficiencia). Cuando se solicitan 8 workers, parte se asignan a cores E que son más lentos, reduciendo el speedup efectivo.

## Fenómeno 5: El tamaño de lote afecta significativamente el rendimiento

**Observación**: en CPU-bound, el mejor tiempo se logró con lote=200; con lote=5000 el tiempo aumentó. En I/O-bound el efecto es aún más dramático: con lote=5000 sobre 3000 ítems, la paralelización desaparece casi por completo.

**Explicación**: el tamaño de lote controla la **granularidad** del trabajo distribuido:

- **Lotes muy chicos**: muchas unidades de trabajo, pero cada una conlleva overhead de despacho. Si el overhead supera al cómputo útil, el rendimiento decae.
- **Lotes muy grandes**: pocas unidades de trabajo, varios workers quedan ociosos. En el caso extremo (1 lote), el paralelismo se vuelve secuencial.
- **Punto óptimo**: depende del tamaño total del dataset, el número de workers y el costo por ítem.

**Implicación práctica**: al diseñar un pipeline paralelo, dimensionar el batch size es una decisión crítica. No existe un valor "correcto" universal — debe ajustarse experimentalmente para cada caso.

## Análisis del error de asumir speedup lineal

La pauta exige discutir este punto explícitamente. Los resultados del experimento B lo evidencian de varias formas:

| Carga | Estrategia | Workers | Speedup observado | Speedup ideal | Eficiencia |
|---|---|---|---|---|---|
| CPU | threads | 2 | ~1.0× | 2× | ~50% (pero ningún beneficio real) |
| CPU | threads | 8 | ~1.0× | 8× | ~12% (GIL serializa) |
| CPU | processes | 4 | ~2.85× | 4× | ~71% |
| CPU | processes | 8 | ~3.46× | 8× | ~43% |
| I/O | threads | 4 | ~1.4× | 4× | ~35% |
| I/O | processes | 4 | ~1.25× | 4× | ~31% |

**Conclusiones del análisis**:

- En **ninguna configuración** el speedup es lineal.
- En **threads + CPU-bound**, no hay speedup en absoluto — el GIL anula la ventaja.
- En **processes + CPU-bound**, hay speedup positivo pero claramente sub-lineal por overhead.
- En **I/O-bound**, ambos paralelos rinden modestamente debido a contención de almacenamiento.

Asumir speedup lineal lleva a malas decisiones de infraestructura: dimensionar mal los recursos, esperar mejoras de rendimiento que no ocurrirán, y no identificar correctamente los cuellos de botella reales (CPU, GIL, I/O o coordinación).

## Limitaciones y observaciones metodológicas

1. **Mediciones de CPU poco confiables para corridas cortas**: igual que en el experimento A, `psutil.cpu_percent` puede reportar 0% en corridas de duración menor al intervalo de muestreo del kernel (~100ms).

2. **I/O sobre SSD compartido**: las mediciones I/O reflejan el comportamiento de un solo dispositivo de almacenamiento. En infraestructura distribuida con múltiples discos o almacenamiento en red, el comportamiento sería distinto.

3. **Cores heterogéneos del Apple M4**: la asignación de workers a cores específicos (P vs E) la decide el scheduler del sistema operativo y no se controló explícitamente. Esto introduce variabilidad en los resultados con configuraciones de 8 workers.

4. **Carga sintética**: tanto el CPU-bound (limpieza de strings) como el I/O-bound (write/read/delete) son cargas representativas pero simplificadas. Pipelines reales pueden mezclar ambos tipos en proporciones variables.

5. **Tamaño del dataset I/O limitado**: 3000 ítems es un número pequeño. Datasets más grandes podrían mostrar patrones distintos, especialmente con tamaños de lote grandes.

## Recomendaciones según escenario

| Escenario | Recomendación |
|---|---|
| Limpieza/transformación de strings o datos numéricos en lotes | `ProcessPoolExecutor` con número de workers ≈ cores P físicos. |
| Lectura/escritura de archivos, llamadas a APIs, consultas a BD | `ThreadPoolExecutor` con número de workers según tolerancia del recurso (filas/conexiones simultáneas). |
| Pipelines mixtos (CPU + I/O) | Considerar dividir el pipeline en etapas, paralelizando cada una con la estrategia óptima. |
| Dataset con tamaño desconocido o variable | Dimensionar el tamaño de lote dinámicamente, asegurando al menos `2×n_workers` lotes para evitar desbalance. |
| Servidor único | Threads/processes locales como en este experimento. |
| Múltiples fuentes de datos | Considerar Dask o frameworks de orquestación que coordinen lectura y procesamiento. |
| Plataforma cloud | Spark o Dask distribuido si el dataset excede capacidad de un nodo; lambdas/funciones serverless para cargas embarrassingly parallel. |
